In [1]:
# Modules
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from IPython.display import HTML
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Plotting figures. Overlaying multiple graphs in a single figure
def plot_solution(xgrid, uvect):
  # Setting parameters related to plotting
  umax = np.max(sol)
  umin = np.min(sol)
  ulength = umax - umin
  umax += 0.05 * ulength
  umin -= 0.05 * ulength
  # t = 0
  plt.plot(xgrid,uvect[0,:],color='#ff8c00')
  # t > 0
  for n in range(1,uvect.shape[0]):
    plt.plot(xgrid,uvect[n,:],color='#00bfff')

  plt.xlabel('x')
  plt.ylabel('u')
  plt.ylim(umin, umax)
  plt.grid('on')
  plt.show()

In [3]:
# Creating animations
# Using the IPython.display module
def plot_animation(xgrid, tgrid, uvect):
  # Configuring plotting-related parameters
  umax = np.max(sol)
  umin = np.min(sol)
  xmax = np.max(xgrid)
  xmin = np.min(xgrid)
  ulength = umax - umin
  utop = umax + 0.1*ulength
  umax += 0.05*ulength
  umin -= 0.05*ulength
  xmid = (xmax+xmin)/2
  xlength = xmax - xmin
  xmax += 0.05*xlength
  xmin -= 0.05*xlength

  # Creating fig and ax objects
  fig, ax = plt.subplots()
  ims = []

  # t = 0
  n = 0
  im, = ax.plot(xgrid, uvect[n, :], color='#ff8c00')
  title = ax.text(xmid, utop, f'time={tgrid[n]:.4f}', ha='center', va='center', fontsize=15)
  ims.append([im, title])

  # t > 0
  for n in range(1, uvect.shape[0]):
    im, = ax.plot(xgrid, uvect[n, :], color='#00bfff')
    title = ax.text(xmid, utop, f'time={tgrid[n]:.4f}', ha='center', va='center', fontsize=15)
    ims.append([im, title])

  # Setting labels for each axis
  ax.set_xlabel(r"$x$", fontsize=15)
  ax.set_ylabel(r"$u$", fontsize=15)
  # Setting the range of the graph
  ax.set_xlim([xmin, xmax])
  ax.set_ylim([umin, umax])
  ax.grid(True)

  # Creating an animation by passing the fig object and ims to ArtistAnimation
  return animation.ArtistAnimation(fig, ims)

In [4]:
def find_maximum(ff, init):
  s0 = np.min(init)
  s1 = np.max(init)
  Ns = 1000
  xs = np.linspace(s0, s1, Ns)
  ys = ff(xs)
  dfs = (ys[1:] - ys[:-1])*Ns/(s1-s0)
  return s0, s1, np.max(np.abs(dfs))

In [5]:
# Lax--Friedrics scheme for the nonlinear conservation law
def conservationlaw_lf(flux, initial, N, T, mu, num):
  # x-grids
  L = 1.0
  h = L/N
  xgrid = np.linspace(0.0, L, N)
  xtmp = xgrid[:-1].copy()

  # initial setteings
  tgrid = np.array([0.0])
  u = initial(xtmp)
  sol = np.copy(np.append(u,u[0]))

  # CFL condition
  smin, smax, abs_df = find_maximum(flux, u)

  # t-grids
  tau = mu*h/abs_df
  lam = tau/h
  nmax = int(T/tau)
  step = int(max(1, nmax/num))

  # iteration
  const = 0.5/lam
  n = 0
  tnow = n*tau
  for n in range(nmax):
    uflux = 0.5*(flux(u) + flux(np.roll(u,1))) - const * (u - np.roll(u,1))
    u = u - lam*(np.roll(uflux,-1) - uflux)
    tnow = (n+1)*tau
    if (n+1)%step==0:
      sol=np.vstack((sol,np.append(u,u[0])))
      tgrid=np.append(tgrid, tnow)

  return xgrid, tgrid, sol

In [ ]:
# Initial data
def initial1(x):
  return 0.2*np.sin(2*np.pi*x) + 1.2

# Flux function
def flux1(s):
  return 0.5*s*s

# Setting parameters
N = 201
T = 2
mu = 1
num = 40

# Computing the nonlinear conservation law
x, tn, sol = conservationlaw_lf(flux1, initial1, N, T, mu, num)

# Displaying figures
plot_solution(x, sol)

# Displaying an animation
anim = plot_animation(x, tn, sol)
# Saving the results as convect.gif (make sure that Google Drive is mounted)
anim.save('/content/drive/MyDrive/Colab Notebooks/fig/ncl.gif', writer='pillow')
rc('animation', html='jshtml')
plt.close()
anim